Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

Ans. 1. The Driver is the main program that creates the SparkSession, divides the work into tasks, and coordinates execution.
     2. The Cluster Manager is responsible for allocating resources and managing the cluster.
     3. Executors are worker processes that execute the tasks assigned by the Driver and return the results.

Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain
processing large datasets?

Ans. Spark does not execute transformations immediately. Instead, it records them and creates a logical execution plan called a DAG (Directed Acyclic Graph). Execution starts only when an action is called. This allows Spark to optimize the workflow, reduce unnecessary computations, and improve performance.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6_Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [2]:
data = [
    (1, "John", "North", "Electronics", 1500, 25, "Premium", "[email protected]", "2025-01-01", 101, "Completed", "High"),
    (2, "Alice", "West", "Clothing", 300, 22, "Premium", "[email protected]", "2025-01-02", 101, "Pending", "Low"),
    (3, "Bob", "East", "Electronics", None, 35, "Basic", None, "2025-01-03", 102, "Completed", "Medium"),
    (4, "", "North", "Electronics", 2500, 28, "Premium", "[email protected]", "2025-01-04", 103, "Completed", "High"),
    (5, "Emma", "South", "Furniture", 4000, 30, "Basic", "[email protected]", "2025-01-05", 104, "Pending", "Low"),
    (1, "John", "North", "Electronics", 1500, 25, "Premium", "[email protected]", "2025-01-01", 101, "Completed", "High")
]

columns = [
    "user_id",
    "username",
    "region",
    "product_category",
    "sale_amount",
    "age",
    "subscription",
    "email",
    "transaction_date",
    "store_id",
    "status",
    "priority"
]

df = spark.createDataFrame(data, columns)

df.show()

+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+
|user_id|username|region|product_category|sale_amount|age|subscription|            email|transaction_date|store_id|   status|priority|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+
|      1|    John| North|     Electronics|       1500| 25|     Premium|[email protected]|      2025-01-01|     101|Completed|    High|
|      2|   Alice|  West|        Clothing|        300| 22|     Premium|[email protected]|      2025-01-02|     101|  Pending|     Low|
|      3|     Bob|  East|     Electronics|       NULL| 35|       Basic|             NULL|      2025-01-03|     102|Completed|  Medium|
|      4|        | North|     Electronics|       2500| 28|     Premium|[email protected]|      2025-01-04|     103|Completed|    High|
|      5|    Emma| South|       Furniture|       4000| 

Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs.
columnar) and why does it matter for performance?

Ans. CSV and Parquet are both file formats used to store data, but they work differently.

CSV is a row-based format where data is stored row by row. It is simple, human-readable, and widely supported.
Parquet is a columnar format where data is stored column by column. It provides better compression and faster query performance.

Parquet is generally preferred in Spark because it reads only the required columns, reduces storage space, and improves processing speed for large datasets.

Q5: Given a DataFrame df, write a query to select the columns product_id and price
where the category is 'Electronics'.

In [3]:
df.filter(df.product_category == "Electronics") \
  .select("user_id", "sale_amount") \
  .show()

+-------+-----------+
|user_id|sale_amount|
+-------+-----------+
|      1|       1500|
|      3|       NULL|
|      4|       2500|
|      1|       1500|
+-------+-----------+



Q6: Write the code to "revise" a DataFrame by renaming the column old_name to
new_name and casting the price column from a String to a Double.

In [4]:
from pyspark.sql.functions import col

df_new = df.withColumnRenamed("username", "new_name") \
           .withColumn("sale_amount", col("sale_amount").cast("double"))

df_new.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- new_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- store_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)



Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker
node fails?

Ans. Spark maintains a Lineage Graph (DAG) that records all transformations performed on the data. If a partition is lost due to a node failure, Spark can recompute the lost data using the transformations stored in the DAG. This provides fault tolerance without storing multiple copies of data.

Q8: Write a query to filter a DataFrame df_orders for rows where the status is
'Completed' AND the amount is greater than 1000.

In [5]:
from pyspark.sql.functions import col

df.filter(
    (col("status") == "Completed") &
    (col("sale_amount") > 1000)
).show()

+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+
|user_id|username|region|product_category|sale_amount|age|subscription|            email|transaction_date|store_id|   status|priority|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+
|      1|    John| North|     Electronics|       1500| 25|     Premium|[email protected]|      2025-01-01|     101|Completed|    High|
|      4|        | North|     Electronics|       2500| 28|     Premium|[email protected]|      2025-01-04|     103|Completed|    High|
|      1|    John| North|     Electronics|       1500| 25|     Premium|[email protected]|      2025-01-01|     101|Completed|    High|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+



Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount
of data loaded into memory.

Ans. Predicate Pushdown is an optimization technique in which filter conditions are pushed down to the data source while reading the file. In Parquet files, Spark reads only the required rows instead of scanning the entire dataset. This reduces disk I/O and improves query performance.

Q10: Write a code snippet to add a new column final_price which is the base_price
multiplied by 1.18 (18% tax).

In [6]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("sale_amount") * 1.18
)

df.show()

+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+-----------+
|user_id|username|region|product_category|sale_amount|age|subscription|            email|transaction_date|store_id|   status|priority|final_price|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+-----------+
|      1|    John| North|     Electronics|       1500| 25|     Premium|[email protected]|      2025-01-01|     101|Completed|    High|     1770.0|
|      2|   Alice|  West|        Clothing|        300| 22|     Premium|[email protected]|      2025-01-02|     101|  Pending|     Low|      354.0|
|      3|     Bob|  East|     Electronics|       NULL| 35|       Basic|             NULL|      2025-01-03|     102|Completed|  Medium|       NULL|
|      4|        | North|     Electronics|       2500| 28|     Premium|[email protected]|      2025-01-04|     103|Com

Q11: What is the difference between Transformations and Actions? Provide two
examples of each.

Ans. Transformations create a new DataFrame or RDD and are evaluated lazily. Examples include filter(), select(), and withColumn().

Actions trigger execution and return results. Examples include show(), count(), and collect().

Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out
any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [7]:
from pyspark.sql.functions import col

df_filtered = df.filter(
    col("user_id").isNotNull()
)

df_filtered.show()

+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+-----------+
|user_id|username|region|product_category|sale_amount|age|subscription|            email|transaction_date|store_id|   status|priority|final_price|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+-----------+
|      1|    John| North|     Electronics|       1500| 25|     Premium|[email protected]|      2025-01-01|     101|Completed|    High|     1770.0|
|      2|   Alice|  West|        Clothing|        300| 22|     Premium|[email protected]|      2025-01-02|     101|  Pending|     Low|      354.0|
|      3|     Bob|  East|     Electronics|       NULL| 35|       Basic|             NULL|      2025-01-03|     102|Completed|  Medium|       NULL|
|      4|        | North|     Electronics|       2500| 28|     Premium|[email protected]|      2025-01-04|     103|Com

Q13: In Spark Architecture, what is the difference between Client Mode and Cluster
Mode?

Ans. In Client Mode, the Driver program runs on the machine from which the application is submitted.

In Cluster Mode, the Driver runs inside the cluster on a worker node.

Client Mode is useful for development and debugging, while Cluster Mode is preferred for production workloads.

Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority
is 'High'.

In [8]:
from pyspark.sql.functions import col

df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
).show()

+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+-----------+
|user_id|username|region|product_category|sale_amount|age|subscription|            email|transaction_date|store_id|   status|priority|final_price|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---------+--------+-----------+
|      1|    John| North|     Electronics|       1500| 25|     Premium|[email protected]|      2025-01-01|     101|Completed|    High|     1770.0|
|      4|        | North|     Electronics|       2500| 28|     Premium|[email protected]|      2025-01-04|     103|Completed|    High|     2950.0|
|      1|    John| North|     Electronics|       1500| 25|     Premium|[email protected]|      2025-01-01|     101|Completed|    High|     1770.0|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+---

Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on
a multi-terabyte dataset?

Ans. show(5) displays only a few rows and is safe for large datasets. collect() brings the entire dataset to the Driver's memory. On large datasets, this can consume a lot of memory and may cause performance issues or application failure. Therefore, show(5) is generally preferred for inspecting data.